# DEPRECATED — NON-CANONICAL
Do not execute this historical spike. Start only with `00_campaign_control.ipynb` and follow `docs/canonical-colab-runbook.md`.

# EdgeGuard-Road PIDNet-S single-image spike
Execution-only notebook. It does not contain data, credentials, checkpoint bytes, metrics, or scientific acceptance thresholds. It uses only the approved sample contained in the fixed PIDNet checkout and the approved non-commercial academic checkpoint decision.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
sys.dont_write_bytecode = True
SUBPROCESS_ENV = {**os.environ, "PYTHONDONTWRITEBYTECODE": "1"}
EDGEGUARD_REPOSITORY_URL = "https://github.com/emrealmaoglu/edgeguard-road.git"
EDGEGUARD_BRANCH = "feat/first-vertical-slice"
EDGEGUARD_EXPECTED_COMMIT = input("Reviewed 40-character EdgeGuard commit: ").strip()
EDGEGUARD_ROOT = Path("/content/edgeguard-road")
if len(EDGEGUARD_EXPECTED_COMMIT) != 40 or any(
    character not in "0123456789abcdef" for character in EDGEGUARD_EXPECTED_COMMIT
):
    raise RuntimeError("Enter the exact reviewed lowercase 40-character commit")
if EDGEGUARD_ROOT.exists():
    raise RuntimeError(f"Refuse stale checkout at {EDGEGUARD_ROOT}; start a fresh runtime")
subprocess.run(
    [
        "git",
        "clone",
        "--branch",
        EDGEGUARD_BRANCH,
        "--single-branch",
        EDGEGUARD_REPOSITORY_URL,
        str(EDGEGUARD_ROOT),
    ],
    check=True,
    env=SUBPROCESS_ENV,
)
edgeguard_commit = subprocess.run(
    ["git", "-C", str(EDGEGUARD_ROOT), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
    env=SUBPROCESS_ENV,
).stdout.strip()
if edgeguard_commit != EDGEGUARD_EXPECTED_COMMIT:
    raise RuntimeError(
        f"EdgeGuard commit mismatch: expected {EDGEGUARD_EXPECTED_COMMIT}, got {edgeguard_commit}"
    )
print({"branch": EDGEGUARD_BRANCH, "verified_commit": edgeguard_commit})
os.chdir(EDGEGUARD_ROOT)

In [ ]:
import importlib

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", ".[colab,dev]"],
    cwd=EDGEGUARD_ROOT,
    check=True,
    env=SUBPROCESS_ENV,
)
SOURCE_ROOT = (EDGEGUARD_ROOT / "src").resolve()
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
importlib.invalidate_caches()
edgeguard = importlib.import_module("edgeguard")

edgeguard_file = Path(edgeguard.__file__).resolve()
if not edgeguard_file.is_relative_to(SOURCE_ROOT):
    raise RuntimeError(f"edgeguard imported from unexpected path: {edgeguard_file}")
print({"edgeguard_module": str(edgeguard_file), "python": sys.executable})
subprocess.run(
    [sys.executable, "-m", "edgeguard", "doctor", "--json"],
    cwd=EDGEGUARD_ROOT,
    check=True,
    env=SUBPROCESS_ENV,
)

## Fixed upstream checkout
This cell checks out only the human-approved official commit under the ignored artifacts tree. It is not vendored or added as a submodule.

In [ ]:
PIDNET_REPOSITORY_URL = "https://github.com/XuJiacong/PIDNet.git"
PIDNET_COMMIT = "4c158cf24ce432f0a8cb43364fae38d93cee0dc3"
PIDNET_CHECKOUT = EDGEGUARD_ROOT / "artifacts/external/pidnet" / PIDNET_COMMIT
PIDNET_CHECKOUT.parent.mkdir(parents=True, exist_ok=True)
if not PIDNET_CHECKOUT.exists():
    subprocess.run(
        ["git", "clone", "--no-checkout", PIDNET_REPOSITORY_URL, str(PIDNET_CHECKOUT)],
        check=True,
        env=SUBPROCESS_ENV,
    )
subprocess.run(
    ["git", "-C", str(PIDNET_CHECKOUT), "checkout", "--detach", PIDNET_COMMIT],
    check=True,
    env=SUBPROCESS_ENV,
)
actual_commit = subprocess.run(
    ["git", "-C", str(PIDNET_CHECKOUT), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
    env=SUBPROCESS_ENV,
).stdout.strip()
assert actual_commit == PIDNET_COMMIT, (actual_commit, PIDNET_COMMIT)
actual_origin = subprocess.run(
    ["git", "-C", str(PIDNET_CHECKOUT), "remote", "get-url", "origin"],
    check=True,
    capture_output=True,
    text=True,
    env=SUBPROCESS_ENV,
).stdout.strip()
checkout_status = subprocess.run(
    ["git", "-C", str(PIDNET_CHECKOUT), "status", "--porcelain=v1"],
    check=True,
    capture_output=True,
    text=True,
    env=SUBPROCESS_ENV,
).stdout.strip()
assert actual_origin == PIDNET_REPOSITORY_URL, (actual_origin, PIDNET_REPOSITORY_URL)
assert not checkout_status, checkout_status
print({"repository": actual_origin, "commit": actual_commit, "clean": True})

## Human-controlled checkpoint upload
The old official file link returned HTTP 404 and automatic retrieval failed on 2026-07-26, while the replacement folder linked by the pinned README remained reachable. Obtain `PIDNet_S_Cityscapes_val.pt` only through that official repository-directed folder, then upload exactly that filename. Its checkpoint-specific license remains OPEN QUESTION. Keep it under the ignored artifacts tree and do not redistribute it. The notebook prints byte size and SHA-256, then automatically compares filename and hash with the reviewed config pin before model loading. No alternative direct URL may be invented.

In [ ]:
from google.colab import files

from edgeguard.config import load_pidnet_spike_config
from edgeguard.serialization import sha256_file

spike_config = load_pidnet_spike_config(EDGEGUARD_ROOT / "configs/pidnet_spike.yaml")
CHECKPOINT_SOURCE_URL = spike_config.checkpoint.source_url
CHECKPOINT_PATH = (
    EDGEGUARD_ROOT / "artifacts/external/checkpoints" / spike_config.checkpoint.filename
)
CHECKPOINT_ACCESS_DATE = "REPLACE_WITH_DOWNLOAD_DATE_YYYY_MM_DD"
SAMPLE_ACCESS_DATE = "REPLACE_WITH_CHECKOUT_ACCESS_DATE_YYYY_MM_DD"

if not CHECKPOINT_PATH.is_file():
    print({"official_replacement_folder": CHECKPOINT_SOURCE_URL})
    uploaded = files.upload()
    if set(uploaded) != {CHECKPOINT_PATH.name}:
        raise RuntimeError(f"Upload only {CHECKPOINT_PATH.name}; received {sorted(uploaded)}")
    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    CHECKPOINT_PATH.write_bytes(uploaded[CHECKPOINT_PATH.name])
actual_checkpoint_sha256 = sha256_file(CHECKPOINT_PATH)
print(
    {
        "filename": CHECKPOINT_PATH.name,
        "source_url": CHECKPOINT_SOURCE_URL,
        "byte_size": CHECKPOINT_PATH.stat().st_size,
        "sha256": actual_checkpoint_sha256,
        "license_status": "OPEN QUESTION",
    }
)
if actual_checkpoint_sha256 != spike_config.checkpoint.sha256:
    raise RuntimeError(
        f"Checkpoint SHA-256 mismatch: expected {spike_config.checkpoint.sha256}, "
        f"got {actual_checkpoint_sha256}"
    )
if CHECKPOINT_ACCESS_DATE.startswith("REPLACE_"):
    raise RuntimeError("Record the actual checkpoint download date")
if SAMPLE_ACCESS_DATE.startswith("REPLACE_"):
    raise RuntimeError("Record the fixed-checkout sample access date")

In [ ]:
command = [
    sys.executable,
    "scripts/run_pidnet_spike.py",
    "--config",
    "configs/pidnet_spike.yaml",
    "--upstream-checkout",
    str(PIDNET_CHECKOUT),
    "--checkpoint",
    str(CHECKPOINT_PATH),
    "--checkpoint-access-date",
    CHECKPOINT_ACCESS_DATE,
    "--sample-access-date",
    SAMPLE_ACCESS_DATE,
    "--output-dir",
    "artifacts/dev/pidnet_spike",
]
completed = subprocess.run(
    command,
    cwd=EDGEGUARD_ROOT,
    env=SUBPROCESS_ENV,
    capture_output=True,
    text=True,
)
if completed.stdout:
    print(completed.stdout)
if completed.returncode != 0:
    if completed.stderr:
        print(completed.stderr, file=sys.stderr)
    raise RuntimeError(f"PIDNet spike failed with exit code {completed.returncode}")
if completed.stderr:
    print(completed.stderr, file=sys.stderr)

In [ ]:
subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=EDGEGUARD_ROOT,
    check=True,
    env=SUBPROCESS_ENV,
)